In [ ]:
# MIT License
#
#@title Copyright (c) 2026 CCAI Community Authors { display-mode: "form" }
#
# Permission is hereby granted, free of charge, to any person obtaining a
# copy of this software and associated documentation files (the "Software"),
# to deal in the Software without restriction, including without limitation
# the rights to use, copy, modify, merge, publish, distribute, sublicense,
# and/or sell copies of the Software, and to permit persons to whom the
# Software is furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL
# THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING
# FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER
# DEALINGS IN THE SOFTWARE.


## Introduction to AI: Predicting Tsunami Alerts from Earthquake Observations

---
**Author**:
[Tarini Bhatnagar](https://www.linkedin.com/in/tarinibhatnagar/), Senior Solutions Architect AI, NVIDIA
> This tutorial is a beginner-friendly walk through the complete machine learning workflow, using real earthquake data from the US Geological Survey. By the end, you will have built and evaluated a model that flags earthquakes likely to trigger a tsunami — while learning every concept from the ground up.

---

## Table of Contents

*   [Overview](#overview)
*   [Climate Impact](#climate-impact)
*   [Background & Prerequisites](#background-and-prerequisites)
*   [Software Requirements](#software-requirements)
*   [Data Description](#data-description)
*   [Methodology](#methodology)
*   [Results & Discussion](#results--discussion)
*   [Summary](#summary)
*   [References](#references)

---
<h2 id="overview">Overview</h2>

This tutorial walks through the full machine learning workflow. By the end, you will be able to:

1. **Look at and understand a real dataset** — inspecting shape, distributions, missing values, and geographic patterns
2. **Define input features and a target variable** — the two ingredients every ML model needs
3. **Create train / validation / test splits** — and understand *why* each split exists
4. **Build a simple baseline model** — a common-sense benchmark that any ML model must beat to be useful
5. **Train a basic ML model** — starting from the simplest classifier
6. **Evaluate model performance** — using metrics appropriate for imbalanced problems
7. **Understand overfitting** — see it happen, and learn how to prevent it
8. **Interpret results and discuss limitations** — including responsible-use considerations

You'll also find interactive cells throughout where you can change values and see how they affect the results, plus challenges to improve model performance yourself.

<h2 id="climate-impact">Climate Impact</h2>

#### Tsunamis are among the deadliest natural hazards

When a large earthquake occurs offshore, it can displace the sea floor and generate a **tsunami** — a series of long-wavelength ocean waves that reach coastlines within minutes and can cause catastrophic damage.

Recent examples:
- **2004 Indian Ocean tsunami:** ~230,000 deaths across 14 countries
- **2011 Tōhoku tsunami (Japan):** ~20,000 deaths
- **2022 Hunga Tonga:** observable waves across the entire Pacific basin

#### Not every large earthquake causes a tsunami

Whether an earthquake triggers a tsunami depends on:
- **Magnitude** — typically must exceed roughly M6.5
- **Depth** — shallow earthquakes displace the sea floor more effectively
- **Location** — must be offshore or coastal
- **Focal mechanism** — vertical fault motion generates tsunamis; horizontal (strike-slip) does not

#### Where does AI fit in?

Operational tsunami warning centres — like the [Pacific Tsunami Warning Center](https://www.tsunami.gov/) — combine rule-based decision procedures with numerical models. ML can support this by:
- **Rapidly triaging incoming events** to focus expert attention
- **Learning historical patterns** — which feature combinations most reliably indicate tsunami risk
- **Providing a probability** alongside the traditional binary yes/no

**Our task in this tutorial:** given the standard earthquake catalog information (magnitude, depth, location, and quality metrics), can we predict which events will be flagged as tsunami-relevant?

> **Note:** This tutorial is educational. A real tsunami warning system involves deep-ocean pressure sensors (DART buoys), tide gauges, hydrodynamic modelling, and expert seismologists. Nothing you build here should be used for real-time warnings.

#### The climate connection

Earthquakes themselves aren't caused by climate change. But climate change **amplifies their impacts**:
- **Rising sea levels** mean the same tsunami now inundates coastal communities more deeply
- **Coastal populations continue to grow**, especially in Asia and the Pacific — the regions with highest tsunami exposure
- **Climate-driven displacement** may push more people into hazard-exposed areas

The ML skills you learn here transfer directly across the natural hazards spectrum.

---

<h2 id="background-and-prerequisites">Background and Prerequisites</h2>

**Estimated time:** ~2 hours  
**Prerequisites:** Basic Python; no prior ML experience required, although beginner knowledge of scikit-learn will be helpful.  
**Run in:** Google Colab (recommended) or any Jupyter environment  

---

<h2 id="software-requirements">Software Requirements</h2>

This notebook was developed in July 2026. If you are running this notebook in Google Colab, you can skip the "pip install pandas..." cell below since the packages are already installed by default. If you are running this notebook in a different environment, uncomment the code below (remove hashtags) to install all required packages.

In [ ]:
#!pip install pandas numpy scikit-learn matplotlib seaborn folium
#print("Ready to import libraries in the next cell")

In [ ]:
# Standard data-science imports
import pandas as pd                    # for handling tables of data
import numpy as np                     # for numerical operations
import matplotlib.pyplot as plt        # for making plots
import seaborn as sns                  # for prettier statistical plots
import warnings
warnings.filterwarnings('ignore')

# Folium for interactive maps
import folium
from folium.plugins import MarkerCluster, HeatMap

# scikit-learn — the standard Python ML library
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Plotting style
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

# For reproducibility — same random seed means everyone gets the same results
SEED = 42
np.random.seed(SEED)

print("All libraries imported successfully")

---

<h2 id="data-description">Data Description</h2>

#### Source: USGS Earthquake Catalog

We use the [USGS Earthquake Catalog](https://earthquake.usgs.gov/earthquakes/search/), accessed via the [FDSN Event Web Service API](https://earthquake.usgs.gov/fdsnws/event/1/). The US Geological Survey maintains the authoritative global earthquake database, updated in near real time.

- **Coverage:** Global, from 1900 to present
- **Contents:** Every located earthquake with magnitude, depth, location, time, and quality-control metadata
- **Access:** Free, no API key, no registration
- **Licence:** Public domain — US Government work
- **What we use:** All M4.5+ earthquakes from 2015 to 2024

#### Key variables

| Column | What it means | Unit |
|---|---|---|
| `time` | When the earthquake happened | UTC timestamp |
| `latitude`, `longitude` | Where the epicentre was | degrees |
| `depth` | How deep beneath the surface | km |
| `mag` | Magnitude (energy released) | Mw scale |
| `nst` | How many seismic stations recorded it | count |
| `gap` | Largest azimuthal gap between stations | degrees |
| `dmin` | Distance to nearest station | degrees |
| `rms` | Location-fit residual | seconds |
| `horizontalError` | Uncertainty in the epicentre location | km |
| `depthError` | Uncertainty in the depth | km |
| `tsunami` | **Target:** did USGS flag this event as tsunami-relevant? | 0 or 1 |

#### Note on this notebook's data

The code cell below tries to fetch live earthquake data directly from the USGS API. In Google Colab this can sometimes fail (network policies, rate limits, or the API being temporarily slow), so we fall back to a backup `usgs_prototype.csv` [you can download here](https://raw.githubusercontent.com/climatechange-ai-tutorials/intro-ai-tsunami-alerts/refs/heads/main/usgs_prototype.csv). That's completely fine — the file contains 8,000 realistic events and the rest of the notebook works identically.

If you'd like to work with your own fresh data from USGS instead of the pre-downloaded file, here is what you can do:

Download the CSV manually from the USGS website.
1. Go to https://earthquake.usgs.gov/earthquakes/search/
2. Set your search parameters — for example:
   Magnitude: minimum 4.5
   Date & Time: 2015-01-01 to 2024-12-31
   Geographic Region: World
3. Under Output Options, select CSV
4. Click Search, then download the CSV that appears
5. Upload the downloaded file to Colab (folder icon 📁 → upload) and rename it to usgs_prototype.csv, or update the filename in the code cell above

In case fetching live data fails, you need `usgs_prototype.csv` alongside the notebook. <br>

#### Uploading the Data File
**In Google Colab:** click the folder icon in the left sidebar, then the upload button, and select the CSV — note that Colab uploads are temporary and vanish when the session ends. <br>
**Running locally:** put the CSV in the same folder as the notebook and start Jupyter from that folder. You'll know it worked when the data-loading cell in Section 4.1 prints either "Fetched from USGS live API" or "Loaded from usgs_prototype.csv" instead of a `FileNotFoundError`. <br>

#### Data Download

#### Fetch data from the USGS API

The USGS provides earthquake data through a free web API. The URL parameters control what data we get back:

```
https://earthquake.usgs.gov/fdsnws/event/1/query?
    format=csv               ← we want CSV back
    &starttime=2015-01-01    ← start date
    &endtime=2024-12-31      ← end date  
    &minmagnitude=4.5        ← only include M4.5+ events
```

We can call this URL directly from `pandas.read_csv`, and it returns a table.

In [ ]:
# ── Fetch the data ───────────────────────────────────────────────────────────
START, END, MINMAG = "2015-01-01", "2024-12-31", 4.5
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?"
    f"format=csv&starttime={START}&endtime={END}&minmagnitude={MINMAG}"
)

print(f"Fetching earthquakes from {START} to {END} (magnitude ≥ {MINMAG})...")
try:
    df = pd.read_csv(url)
    print(f"Fetched {len(df):,} earthquakes from the USGS live API")
except Exception as e:
    print(f"Live API unavailable ({type(e).__name__}). Using the prototype file.")
    df = pd.read_csv("usgs_prototype.csv")
    print(f"Loaded {len(df):,} rows from usgs_prototype.csv")

df.head(3)

Note: If the above code results in an error, [download a backup of the usgs_prototype.csv file here](https://raw.githubusercontent.com/climatechange-ai-tutorials/intro-ai-tsunami-alerts/refs/heads/main/usgs_prototype.csv) or follow the [instructions above](#data-description) to download the data directly from the USGS website.


#### Data Exploration and Preprocessing

Before doing any modelling, we need to really understand what we have. Data scientists often say **"time spent understanding your data is never wasted"** — this section might feel long, but every step here is what a professional would do. High quaHigh-quality data - clean, accurate, and relevant is key to AI success



#### How big is the dataset?

The `.shape` attribute tells us the size — a pair of numbers `(rows, columns)`.

In [ ]:
print(f"Number of rows (earthquakes): {df.shape[0]:,}")
print(f"Number of columns (variables): {df.shape[1]}")
print(f"\nDate range: {df['time'].min()[:10]} → {df['time'].max()[:10]}")
print(f"That's about {(pd.to_datetime(df['time'].max()) - pd.to_datetime(df['time'].min())).days / 365:.1f} years of data")

#### What are the columns?

`.dtypes` shows the type of each column. Numerical columns can be used directly by ML models; text ("object") columns need special handling.

In [ ]:
print("All columns in the dataset:")
print(df.dtypes)

#### Are there missing values?

Real-world datasets almost always have some missing values. Ignoring them causes bugs later, so we check up front.

In [ ]:
missing = df.isnull().sum()
missing_nonzero = missing[missing > 0].sort_values(ascending=False)

if len(missing_nonzero) == 0:
    print("✅ No missing values anywhere in the dataset")
else:
    print("Columns with missing values:")
    print(missing_nonzero.to_string())
    print(f"\nTotal missing values: {missing_nonzero.sum():,} ({missing_nonzero.sum()/df.size*100:.2f}% of all cells)")

#### Summary statistics for the numeric columns

`.describe()` gives us the count, mean, standard deviation, min, and quartiles of every numeric column. This is a fast way to spot unusual values or scales.

In [ ]:
main_cols = ['latitude', 'longitude', 'depth', 'mag', 'nst', 'gap', 'tsunami']
df[main_cols].describe().round(2)

**What to notice:**

- **`mag`** — minimum is 4.5 (that's the filter we applied). Maximum is around 8-9 (the largest earthquakes in the past decade).
- **`depth`** — ranges from about 0 km (surface) to several hundred km deep. Most are shallow.
- **`nst`** — number of stations that recorded each event, ranges from a handful to over a hundred.
- **`tsunami`** — mean is around 0.03, meaning about 3% of events triggered a tsunami flag. We'll look at this more closely next.

#### The target variable — how balanced is it?

The `tsunami` column is what we want to predict. Let's see how the two values are distributed.

In [ ]:
tsunami_counts = df['tsunami'].value_counts()
total = len(df)

print("Tsunami flag distribution:")
print(f"  0 (no tsunami):  {tsunami_counts.get(0, 0):>6,}  ({tsunami_counts.get(0, 0)/total*100:5.1f}%)")
print(f"  1 (tsunami):     {tsunami_counts.get(1, 0):>6,}  ({tsunami_counts.get(1, 0)/total*100:5.1f}%)")
print()
print("→ Only a small fraction of earthquakes trigger tsunamis.")
print("  This is called an IMBALANCED classification problem.")
print("  We'll discuss what that means in the next section.")

---
#### Concept Break — Understanding Imbalanced Data and Metrics

Before we go further, let's pause and explain three important concepts that will come up over and over in this tutorial: **imbalanced data**, **precision**, and **recall**.

#### 1. What is imbalanced data?

A classification dataset is **imbalanced** when one class is much more common than the other. Our tsunami dataset is highly imbalanced:

- Roughly 97 out of every 100 earthquakes → no tsunami
- Only about 3 out of every 100 → tsunami

**Why does this matter?** Because a lazy model can get very high accuracy just by predicting "no tsunami" for every single event — it would be right 97% of the time! But that model is completely useless — it would never warn anyone about a real tsunami.

This is why we need better metrics than just accuracy for imbalanced problems.

#### 2. A simple example: the fire alarm

Imagine your school installed a new smoke detector. Over the course of a year, there were **10 real fires**. The detector alerted **50 times** total. Of those 50 alerts, only **8 were real fires** — the other 42 were false alarms (from burnt toast, steam from showers, etc.). And 2 real fires happened when the detector stayed silent.

Let's translate this into ML language:

|  | Detector said "FIRE!" | Detector stayed silent |
|---|---|---|
| **Real fire happened** | 8 (True Positives) | 2 (False Negatives) |
| **No real fire** | 42 (False Positives) | ...many other times |

#### 3. What is precision?

> **Precision** = "When the detector alerted, how often was there really a fire?"
>
> **Precision = True Positives / (True Positives + False Positives)**
>
> In our example: 8 / (8 + 42) = **16%**

Low precision means lots of false alarms. People will start ignoring the detector.

#### 4. What is recall?

> **Recall** = "Of all the real fires that happened, how many did the detector catch?"
>
> **Recall = True Positives / (True Positives + False Negatives)**
>
> In our example: 8 / (8 + 2) = **80%**

Low recall means missed events. Some real fires went undetected.

#### 5. The trade-off

These two metrics **usually pull in opposite directions**:

- If we make the detector more sensitive → catches more real fires (higher recall) → but more false alarms (lower precision)
- If we make it less sensitive → fewer false alarms (higher precision) → but misses more real fires (lower recall)

**For our tsunami problem:**
- **Missing a real tsunami is catastrophic** — people die. So we want high **recall**.
- **False alarms are bad but survivable** — costly and erode trust, but no one dies from an unnecessary evacuation.

So for tsunami warning, we usually prioritize recall over precision.

#### 6. F1 score

Since precision and recall trade off, we sometimes want a single number that combines them. The **F1 score** is the "harmonic mean" of precision and recall — it's high only when *both* are high.

Don't worry about the math — just know that F1 gives us a single balanced score.

---

Now let's continue exploring the data. When we come to model evaluation in Part 9, you'll see these metrics in action.


#### 7. AUC-ROC

What AUC-ROC actually measures. <br>
Recall what precision and recall told us: <br>

Precision: when the model shouts "TSUNAMI!", how often is it right?
Recall: Of the real tsunamis, how many does the model catch?

Both of those depend on where you set the threshold. The model outputs a probability between 0 and 1 for each earthquake. We only convert that to a "yes tsunami / no tsunami" prediction when we pick a cutoff — usually 0.5 by default.
But 0.5 is arbitrary. If we lowered the threshold to 0.3, we'd catch more real tsunamis (higher recall) but issue more false alarms (lower precision). If we raised it to 0.8, we'd have fewer false alarms (higher precision) but miss more real tsunamis (lower recall).
AUC-ROC asks a deeper question: ignore any specific threshold. How good is the model at ranking earthquakes from "most likely tsunami" to "least likely tsunami"?

#### Visualize the Data

Numbers are useful but plots reveal patterns you would otherwise miss.

#### An interactive world map of earthquakes and tsunamis

We use the **Folium** library to create a fully interactive world map. You can zoom, pan, and click each earthquake to see its details.

- **Blue circles** = earthquakes that did NOT trigger a tsunami
- **Red circles** = earthquakes that were flagged as tsunami-triggering

Small blue dots are clustered together so the map stays fast even with thousands of events. Red tsunami-triggering events are shown individually so you can click each one.

In [ ]:
# ── Create the interactive Folium map ────────────────────────────────────────
# Center the map at latitude 20 (a bit above the equator) so we see both hemispheres
world_map = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB positron",   # a clean, minimal basemap
    world_copy_jump=True,
)

# ── Non-tsunami earthquakes: cluster them so the map stays responsive ────────
# MarkerCluster groups nearby markers into a numbered bubble at low zoom levels,
# and expands them when you zoom in.
non_tsunami_cluster = MarkerCluster(name="Non-tsunami earthquakes",
                                     control=True).add_to(world_map)

# Sample down to 2000 non-tsunami events to keep the map fast
# (the full 8000+ would be slow to render in the browser)
np.random.seed(SEED)
non_tsunami = df[df['tsunami'] == 0]
if len(non_tsunami) > 2000:
    non_tsunami = non_tsunami.sample(2000, random_state=SEED)

for _, row in non_tsunami.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='steelblue',
        fill=True, fill_opacity=0.5,
        popup=f"M{row['mag']:.1f} at depth {row['depth']:.0f} km",
    ).add_to(non_tsunami_cluster)

# ── Tsunami-triggering earthquakes: show individually, all in red ────────────
tsunami_layer = folium.FeatureGroup(name="🌊 Tsunami-triggering earthquakes",
                                     show=True).add_to(world_map)

for _, row in df[df['tsunami'] == 1].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='darkred',
        fill=True, fill_color='crimson', fill_opacity=0.85,
        weight=1.5,
        popup=folium.Popup(
            f"<b>Tsunami event</b><br>"
            f"Magnitude: {row['mag']:.1f}<br>"
            f"Depth: {row['depth']:.0f} km<br>"
            f"Location: {row['place']}<br>"
            f"Date: {row['time'][:10]}",
            max_width=250,
        ),
    ).add_to(tsunami_layer)

# Add a layer control so users can toggle each layer on/off
folium.LayerControl(collapsed=False).add_to(world_map)

# Display the map inline in the notebook
world_map

**What to explore:**

- Zoom into different regions using the `+` and `−` buttons on the map (or scroll)
- Click any red circle to see the magnitude, depth, and location of that tsunami event
- Use the layer control (top right) to toggle blue non-tsunami events on/off
- Try zooming into the Pacific — you'll see earthquakes tracing the **Ring of Fire** exactly along plate boundaries

**Take a moment now:** find the country closest to where you live on the map. Are there many earthquakes there? Any tsunami-triggering ones? What might that mean for your local hazard exposure?

#### Distribution of magnitude and depth

Two histograms showing how magnitudes and depths are distributed. Understanding these distributions will help us pick good features later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df['mag'], bins=np.arange(4.5, 9.5, 0.2),
             color='#e67e22', alpha=0.75, edgecolor='white')
axes[0].set_yscale('log')  # log scale because small earthquakes are MUCH more common
axes[0].set_title("Distribution of Earthquake Magnitude\n(y-axis is log scale)")
axes[0].set_xlabel("Magnitude (Mw)")
axes[0].set_ylabel("Number of events (log)")

axes[1].hist(df['depth'].dropna(), bins=50,
             color='#8e44ad', alpha=0.75, edgecolor='white')
axes[1].axvline(70, color='black', ls='--', alpha=0.7,
                label='70 km (shallow/deep boundary)')
axes[1].set_title("Distribution of Earthquake Depth")
axes[1].set_xlabel("Depth (km)")
axes[1].set_ylabel("Number of events")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Observations:")
print(f"  • Magnitude: median = {df['mag'].median():.1f}, max = {df['mag'].max():.1f}")
print(f"  • Depth:     median = {df['depth'].median():.1f} km, max = {df['depth'].max():.0f} km")
print(f"  • Fraction of shallow (<70 km) earthquakes: {(df['depth']<70).mean()*100:.1f}%")

#### Tsunami rate by magnitude — the key predictor

This chart shows what fraction of earthquakes in each magnitude range triggered a tsunami. This is arguably the most important plot in the whole notebook — it tells us which feature will matter most.

In [ ]:
df_temp = df.copy()
df_temp['mag_bin'] = pd.cut(df_temp['mag'], bins=[4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 10])
tsunami_rate = df_temp.groupby('mag_bin', observed=True)['tsunami'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(tsunami_rate)), tsunami_rate.values,
              color='#c0392b', alpha=0.8, edgecolor='white')

for bar, val in zip(bars, tsunami_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontsize=10)

ax.set_xticks(range(len(tsunami_rate)))
ax.set_xticklabels([str(x) for x in tsunami_rate.index], rotation=30, ha='right')
ax.set_title("Tsunami Rate by Magnitude — Big Earthquakes Are Much More Likely to Trigger Tsunamis",
             fontsize=11)
ax.set_xlabel("Magnitude range")
ax.set_ylabel("% of events that triggered a tsunami")
plt.tight_layout()
plt.show()

print("Takeaway:")
print("  • Below M5.5: tsunamis almost never happen (<1%)")
print("  • Between M6.5-7.0: about 1 in 4 events trigger a tsunami")
print("  • Above M7.0: majority trigger tsunamis")
print("  → Magnitude will clearly be a strong predictor for our model")

#### How does depth affect the tsunami rate?

Let's do the same analysis for depth. Shallow earthquakes should be more likely to generate tsunamis.

In [ ]:
df_temp = df.copy()
df_temp['depth_bin'] = pd.cut(df_temp['depth'], bins=[0, 30, 70, 150, 300, 700])
depth_rate  = df_temp.groupby('depth_bin', observed=True)['tsunami'].mean() * 100
depth_count = df_temp.groupby('depth_bin', observed=True).size()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(depth_rate)), depth_rate.values,
              color='#8e44ad', alpha=0.8, edgecolor='white')

for bar, val, cnt in zip(bars, depth_rate.values, depth_count.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%\n(n={cnt:,})', ha='center', fontsize=9)

ax.set_xticks(range(len(depth_rate)))
ax.set_xticklabels(['0-30 km\n(very shallow)', '30-70 km\n(shallow)',
                    '70-150 km\n(intermediate)', '150-300 km\n(deep)',
                    '300+ km\n(very deep)'], fontsize=9)
ax.set_title("Tsunami Rate by Depth — Shallow Earthquakes Are More Dangerous", fontsize=11)
ax.set_xlabel("Depth range")
ax.set_ylabel("% of events that triggered a tsunami")
plt.tight_layout()
plt.show()

print("Takeaway: Very shallow earthquakes (<70 km) are MUCH more likely to trigger tsunamis")
print("than deep ones. This matches the physics — deep earthquakes don't move the sea floor.")

#### Try it yourself: explore different magnitude thresholds

Now it's your turn. The cell below prints statistics about earthquakes above a magnitude threshold you can change. Try different values and observe how the tsunami rate changes.

**Suggested experiments:**
- `MAG_THRESHOLD = 5.0` — see the baseline tsunami rate for smaller earthquakes
- `MAG_THRESHOLD = 6.5` — see how the rate jumps for larger events
- `MAG_THRESHOLD = 7.5` — see what happens at the top of the scale

In [ ]:
# ──  CHANGE THIS VALUE AND RE-RUN THE CELL ──────────────────────────────
MAG_THRESHOLD = 6.0     # Try 5.0, 6.5, 7.0, 7.5 — see how the tsunami rate changes
# ───────────────────────────────────────────────────────────────────────────

subset = df[df['mag'] >= MAG_THRESHOLD]
n_events    = len(subset)
n_tsunamis  = int(subset['tsunami'].sum())
rate        = subset['tsunami'].mean() * 100 if n_events > 0 else 0

print(f"Earthquakes with magnitude ≥ {MAG_THRESHOLD}:")
print(f"  Total events:          {n_events:,}")
print(f"  Tsunami-triggering:    {n_tsunamis:,}")
print(f"  Tsunami rate:          {rate:.1f}%")
print()
print("Compare this to the full dataset:")
print(f"  All events:            {len(df):,}")
print(f"  Full tsunami rate:     {df['tsunami'].mean()*100:.1f}%")
print()
print("→ What does this tell you about how magnitude relates to tsunami risk?")

#### Reflection 1

> **Q1.** Looking at the world map, where do most earthquakes cluster? Do these locations match what you know about plate tectonics?
>
> **Q2.** Combining the magnitude chart (4.9) and depth chart (4.10), which combination of magnitude and depth would you expect to be **most likely** to trigger a tsunami? Which combination would be **least likely**?
>
> **Q3.** Only ~3% of earthquakes trigger tsunamis (imbalanced!). If our model simply predicted "no tsunami" for every single event, what would its accuracy be? Why is that not useful?

**Q1:**

**Q2:**

**Q3:**

---

## Methodology

#### Define Features and Target

> **Learning objective:** *Define input features and the target variable.*

Every supervised ML model needs two things:

- **Target (y):** what we're predicting — the `tsunami` flag (0 or 1)
- **Features (X):** the information we use to make the prediction — magnitude, depth, location, and so on. These are called the predicting variables.

#### Step 1: Handle missing values

Handling missing data appropriately is essential for accurate analysis and robust machine learning. The best strategy depends on your dataset's size and the nature of the missingness. Common methods range from simple deletions and central statistics to advanced predictive algorithms and time-series propagation. This catalog has a small number of missing values in some quality-control columns. We fill each of them with the median (the middle value) of that column — a simple but sensible baseline.

In [ ]:
df_clean = df.copy()

numeric_impute = ['nst', 'gap', 'dmin', 'rms',
                  'horizontalError', 'depthError', 'magError', 'magNst']
for col in numeric_impute:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

remaining = df_clean[numeric_impute].isnull().sum().sum()
print(f"Missing values after imputation: {remaining}")

#### Step 2: Understanding cyclic encoding for latitude and longitude

This next part is worth understanding carefully — it's one of those small but important techniques that experienced ML practitioners take for granted, but which trips up beginners.

#### The problem: longitude wraps around

Longitude runs from -180° to +180°. But here's the thing: **longitude 179° and longitude -179° are geographically almost the same place** — just on opposite sides of the International Date Line, near Fiji.

If we feed raw longitude to a machine learning model, it will see:
- Longitude 179 and longitude -179 as **358 units apart** (very different!)
- Longitude 0 (London) and longitude 180 (Fiji) as **180 units apart** (a bit different!)

But the geographic reality is:
- 179 and -179 are **almost the same location** (~2° apart)
- 0 (London) and 180 (Fiji) are **on opposite sides of the world**

This is called the **wrap-around problem** or the **circular data problem**. The model can misunderstand which locations are close together.

#### The solution: sine and cosine

Instead of using longitude directly, we compute two new features:

- `lon_sin = sin(longitude in radians)`
- `lon_cos = cos(longitude in radians)`

Each longitude value becomes a **pair** of numbers between -1 and +1. Together, these two numbers correspond to a unique point on the unit circle. And crucially: **nearby longitudes on the globe correspond to nearby points on this circle**, even across the -180/+180 wrap.

The same trick works for latitude, month, day of week, hour of day — anything that "wraps around".

#### A concrete example

Let's verify this actually works with a small experiment.

In [ ]:
# Three points on Earth:
locations = {
    "Fiji (east side)":    179.0,
    "Fiji (west side)":   -179.0,
    "London":                0.0,
}

print("Raw longitude 'distances' (subtracting the numbers):")
print(f"  Fiji-east to Fiji-west: {abs(179.0 - (-179.0)):.1f} units")
print(f"  London    to Fiji-east: {abs(0.0 - 179.0):.1f} units")
print()
print("→ Raw math says Fiji-east and Fiji-west are 358 units apart —")
print("  further than London is from Fiji. That's obviously wrong.")
print()

# Now compute the cyclic-encoded distances
def cyclic_distance(lon_a, lon_b):
    a_sin, a_cos = np.sin(np.radians(lon_a)), np.cos(np.radians(lon_a))
    b_sin, b_cos = np.sin(np.radians(lon_b)), np.cos(np.radians(lon_b))
    return np.sqrt((a_sin - b_sin)**2 + (a_cos - b_cos)**2)

print("With cyclic (sin, cos) encoding:")
print(f"  Fiji-east to Fiji-west: {cyclic_distance(179.0, -179.0):.4f} — small (correct!)")
print(f"  London    to Fiji-east: {cyclic_distance(0.0, 179.0):.4f} — large (correct!)")
print()
print("→ The two Fiji points are now close together, and London is far away —")
print("  which matches physical reality on the sphere.")

#### A picture is worth a thousand words

Here's a visualisation showing how longitudes map onto the unit circle. Notice how the values wrap around smoothly — there's no jump from -180 to +180.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left panel: longitude on a number line (the WRONG representation) ────────
lons_sample = np.array([-179, -90, 0, 90, 179])
labels = ["Fiji-west\n(-179)", "New Orleans\n(-90)", "London\n(0)",
          "Bangladesh\n(90)", "Fiji-east\n(179)"]
colours = plt.cm.viridis(np.linspace(0.15, 0.85, len(lons_sample)))

axes[0].scatter(lons_sample, np.zeros_like(lons_sample), s=200, c=colours, zorder=3)
for lon, lab, c in zip(lons_sample, labels, colours):
    axes[0].annotate(lab, (lon, 0), xytext=(0, 15), textcoords='offset points',
                     ha='center', fontsize=9)
axes[0].axhline(0, color='grey', lw=0.5)
axes[0].set_xlim(-190, 190)
axes[0].set_ylim(-1, 1)
axes[0].set_xlabel("Raw longitude value")
axes[0].set_title("Raw longitude on a number line\n(Fiji-west and Fiji-east look FAR apart — wrong!)")
axes[0].set_yticks([])

# ── Right panel: same points on the unit circle (the CORRECT representation) ─
lons_dense = np.linspace(-180, 180, 361)
axes[1].plot(np.sin(np.radians(lons_dense)), np.cos(np.radians(lons_dense)),
             'grey', lw=0.8, alpha=0.6)

for lon, lab, c in zip(lons_sample, labels, colours):
    x = np.sin(np.radians(lon))
    y = np.cos(np.radians(lon))
    axes[1].scatter(x, y, s=200, c=[c], zorder=3, edgecolor='white', lw=2)
    axes[1].annotate(lab, (x, y), xytext=(15, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=9)
axes[1].set_xlim(-1.8, 2.3)
axes[1].set_ylim(-1.5, 1.5)
axes[1].set_aspect('equal')
axes[1].set_xlabel("lon_sin")
axes[1].set_ylabel("lon_cos")
axes[1].set_title("The same longitudes as (sin, cos) coordinates\n(Fiji-west and Fiji-east are neighbours!)")

plt.tight_layout(); plt.show()

print("→ On the LEFT (raw longitude), Fiji-west and Fiji-east are at opposite ends.")
print("→ On the RIGHT (cyclic encoding), they sit next to each other on the circle.")
print("  The second picture is what a machine learning model 'sees' when we use sin/cos.")

#### Should we do the same for latitude?

**Latitude does NOT wrap around** — it runs from -90 (South Pole) to +90 (North Pole), and these two poles are truly far apart, not close together. So strictly, latitude does not need cyclic encoding.

However, we still apply the same sin/cos transformation to latitude in this tutorial for two reasons:
1. **Consistency** — treating both coordinates the same way is simpler
2. **Physical meaningfulness** — `cos(latitude)` naturally captures the "distance from the equator", which relates to the geometry of the Earth's surface

Now let's apply this to our data.

In [ ]:
# ── Cyclic encoding for latitude and longitude ──────────────────────────────
df_clean['lat_sin'] = np.sin(np.radians(df_clean['latitude']))
df_clean['lat_cos'] = np.cos(np.radians(df_clean['latitude']))
df_clean['lon_sin'] = np.sin(np.radians(df_clean['longitude']))
df_clean['lon_cos'] = np.cos(np.radians(df_clean['longitude']))

# ── Also create a boolean flag for shallow earthquakes ──────────────────────
# Shallow earthquakes are much more likely to generate tsunamis (we saw this
# in Section 4.10), so making this explicit helps the model
df_clean['shallow'] = (df_clean['depth'] < 70).astype(int)

# Look at the first few rows of the new features
df_clean[['latitude', 'lat_sin', 'lat_cos', 'longitude', 'lon_sin', 'lon_cos', 'shallow']].head()

#### Step 3: Choose our features and target

Now we assemble the final list of features (predicting variables) that will feed the model.

In [ ]:
FEATURES = [
    'mag',              # magnitude — should be the strongest predictor
    'depth',            # depth in km
    'shallow',          # boolean flag for shallow earthquake
    'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos',  # location (cyclic-encoded)
    'nst',              # number of stations that recorded it
    'gap',              # the largest angle between two adjacent seismic stations measuring an earthquake, relative to the epicenter
    'horizontalError',  # location uncertainty
    'depthError',       # depth uncertainty
]
TARGET = 'tsunami'

print(f"Features ({len(FEATURES)}):")
for f in FEATURES:
    print(f"  • {f}")
print(f"\nTarget: {TARGET}")

#### Step 4: Split data into Train / Validation / Test sets

> **Learning objective:** *Create train/validation/test splits.*

#### Why do we split the data?

We can't evaluate a model on the same data we trained it on — the model has "seen" those examples and will look artificially good. We hold out data the model has never seen.

Therefore, we divide the data into three splits, each with a different job:

| Split | Purpose | Size |
|---|---|---|
| **Training** | The model learns from this data | ~60% |
| **Validation** | Compare models, tune parameters | ~20% |
| **Test** | Final unbiased performance — look at only once | ~20% |

#### What is "stratification" and why do we need it here?

Because tsunamis are so rare, a random split might accidentally put most (or all!) of the tsunami events in one split by chance. Then our evaluation would be unstable — the numbers would jump around each time we re-ran the notebook.

**Stratified splitting** solves this. It splits each class separately, so the tsunami rate is nearly identical in all three splits. We pass `stratify=y` to `train_test_split`.

In [ ]:
X = df_clean[FEATURES]
y = df_clean[TARGET]

# ── First split: separate out the test set (20%) ────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,       # 20% goes to test
    random_state=SEED,    # for reproducibility
    stratify=y            # preserve the class balance
)

# ── Second split: split remaining 80% into 75%/25% → train/val ──────────────
# 75% of 80% = 60% of the original data becomes training
# 25% of 80% = 20% of the original data becomes validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp
)

print(f"{'Split':12s} {'Rows':>6s} {'Tsunamis':>10s} {'Rate':>8s}")
print("-" * 42)
for name, X_, y_ in [("Train", X_train, y_train),
                     ("Validation", X_val, y_val),
                     ("Test", X_test, y_test)]:
    print(f"{name:12s} {len(X_):6d} {int(y_.sum()):10d} {y_.mean()*100:7.2f}%")

print("\n→ Tsunami rate is nearly identical across all three splits — stratification worked.")
print("\n→ Now, we will keep our Test set separate, only to be used when evaluating our model on unseen data.")


#### Step 5: Build a Simple Baseline Model

> **Learning objective:** *Build a simple baseline model.*

#### Why bother with a baseline?

Before training any fancy ML model, we ask: **how well would a really simple approach do?**

If our sophisticated model can't beat common sense, then it's not actually adding value — we might as well just use the simple approach.

#### Two baselines to try

1. **Always predict "no tsunami"** — this gets ~97% accuracy just by ignoring the data. We include it here to prove that accuracy is a misleading metric for imbalanced problems.

2. **A simple rule from seismology:** flag events where **magnitude ≥ 7.0 AND depth < 70 km**. This captures what we saw in Section 4 — big shallow earthquakes are the dangerous ones. If our ML model can't beat this rule, we should just use the rule.

So, let us now test our baseline models.

In [ ]:
# ── Baseline 1: always predict 0 (no tsunami) ────────────────────────────────
y_baseline_zero = np.zeros(len(y_val), dtype=int)

# ── Baseline 2: simple magnitude-and-depth rule ─────────────────────────────
y_baseline_rule = ((X_val['mag'] >= 7.0) & (X_val['shallow'] == 1)).astype(int).values

# ── Helper function to print all four metrics for a set of predictions. Instead of re-writing and duplicating code, we will call this helper function everytime we need to evaluate metrics──────
def print_metrics(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    print(f"{name:42s}  acc={acc:.3f}  precision={prec:.3f}  recall={rec:.3f}  F1={f1:.3f}")

print("BASELINE PERFORMANCE ON VALIDATION SET")
print("=" * 95)
print_metrics("Always predict 'no tsunami'",       y_val, y_baseline_zero)
print_metrics("Rule: M >= 7.0 AND depth < 70 km", y_val, y_baseline_rule)

print()
print("→ Always-predict-zero has HIGH accuracy (~97%) but ZERO recall — useless.")
print("→ The simple rule gets some real recall. THIS is the honest bar we must beat.")
print("→ Remember, how we came to the conclusion of prioritizing recall over precision for Tsunamis? ")

#### Try it yourself: tune the rule-based baseline

The rule above uses `magnitude ≥ 7.0` and `depth < 70 km`. What if we changed those thresholds? Try different values and see what happens to precision and recall.

In [ ]:
# ── CHANGE THESE VALUES AND RE-RUN ─────────────────────────────────────
MAG_CUTOFF   = 7.0    # Try 5.5, 6.0, 6.5, 7.5
DEPTH_CUTOFF = 70     # Try 30, 50, 100, 200
# ──────────────────────────────────────────────────────────────────────────

y_custom_rule = ((X_val['mag'] >= MAG_CUTOFF) &
                 (X_val['depth'] < DEPTH_CUTOFF)).astype(int).values

n_alerts = y_custom_rule.sum()
label = f"Rule: M >= {MAG_CUTOFF} AND depth < {DEPTH_CUTOFF} km"

print(f"With these thresholds, the rule flags {n_alerts:,} events out of {len(y_val):,}.")
print()
print_metrics(label, y_val, y_custom_rule)
print()
print("Watch how precision and recall trade off:")
print("  • LOWER magnitude cutoff → more alerts → higher recall but lower precision")
print("  • HIGHER magnitude cutoff → fewer alerts → lower recall but higher precision")
print("  • SHALLOWER depth cutoff → fewer alerts (stricter) → similar trade-off")

#### Step 6: Train a Basic ML Model

> **Learning objective:** *Train a basic ML model.*

#### Our first model: Logistic Regression

Logistic regression is the simplest classifier that actually learns from data. It takes a weighted combination of features and squashes the result into a probability between 0 and 1. Despite its simplicity, it's often surprisingly hard to beat on tabular data.

#### One preprocessing step: scaling

Logistic regression works best when features are on similar scales. Our features range widely (some are in the range -1 to 1, others go up to 40 or more). We **standardize** them: subtract the mean and divide by the standard deviation so every feature has mean 0 and standard deviation 1.

**Important:** we fit the scaler *only on the training data*, then apply the same transformation to validation and test. Fitting the scaler on the full dataset would leak information from the test set into training.

In [ ]:
# ── Scale the features ───────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform on training
X_val_scaled   = scaler.transform(X_val)         # transform only on val
X_test_scaled  = scaler.transform(X_test)        # transform only on test

# ── Train the model ──────────────────────────────────────────────────────────
# class_weight='balanced' tells the model to care extra about the rare class.
# Without this, it might just learn to always predict "no tsunami" (like Baseline 1).
model = LogisticRegression(
    class_weight='balanced',
    max_iter=500,
    random_state=SEED,
)
model.fit(X_train_scaled, y_train)

print("✅ Model trained")
print(f"   Features used: {model.n_features_in_}")

---

## Results & Discussion


#### **Evaluate the Model**

> **Learning objective:** *Evaluate model performance.*

Time to see how well our model does. We use the metrics we introduced in the "Concept Break" earlier: precision, recall, F1, and AUC-ROC.

Quick reminder of what each one means:

| Metric | Meaning |
|---|---|
| **Accuracy** | Overall correctness — misleading when classes are imbalanced |
| **Precision** | Of events we flagged, how many were real tsunamis? |
| **Recall** | Of real tsunamis, how many did we catch? |
| **F1** | Balanced summary of precision and recall |
| **AUC-ROC** | Ranking quality across all decision thresholds |

For tsunami warnings, we care most about **recall** — a missed real event costs lives.

In [ ]:
# ── Predict on the validation set ────────────────────────────────────────────
y_pred  = model.predict(X_val_scaled)              # 0 or 1 predictions
y_proba = model.predict_proba(X_val_scaled)[:, 1]  # probability of tsunami

print("MODEL PERFORMANCE ON VALIDATION SET")
print("=" * 55)
print(f"Accuracy:  {accuracy_score(y_val, y_pred):.3f}")
print(f"Precision: {precision_score(y_val, y_pred):.3f}")
print(f"Recall:    {recall_score(y_val, y_pred):.3f}")
print(f"F1 score:  {f1_score(y_val, y_pred):.3f}")
print(f"AUC-ROC:   {roc_auc_score(y_val, y_proba):.3f}")

print("\nDetailed classification report:")
print(classification_report(y_val, y_pred,
      target_names=["No tsunami", "Tsunami"], digits=3)) #This function in machine learning evaluates the performance of a classification model. It generates a text-based or dictionary summary of key prediction metrics—Precision, Recall, F1-score, and Support—broken down by each individual category (class)

#### Try it yourself: change the decision threshold

By default, the model predicts "tsunami" when its estimated probability is above **0.5**. But 0.5 is just a convention — you can change it to be more or less strict.

- **Lower threshold** (e.g. 0.3): the model becomes more sensitive — catches more real tsunamis (higher recall) but issues more false alarms (lower precision)
- **Higher threshold** (e.g. 0.8): the model becomes more conservative — fewer false alarms (higher precision) but misses more real tsunamis (lower recall)

Change the value below and see how each metric responds. **Which threshold would you choose for a tsunami warning system, and why?**

In [ ]:
# ── CHANGE THIS VALUE AND RE-RUN ─────────────────────────────────────────
DECISION_THRESHOLD = 0.5    # Try 0.2, 0.3, 0.5, 0.7, 0.9
# ────────────────────────────────────────────────────────────────────────────

# Apply the custom threshold
y_pred_thresh = (y_proba >= DECISION_THRESHOLD).astype(int)

print(f"Model performance at threshold = {DECISION_THRESHOLD}")
print("=" * 55)
print(f"Alerts issued: {y_pred_thresh.sum()} out of {len(y_val)} events")
print()
print_metrics(f"Threshold {DECISION_THRESHOLD}", y_val, y_pred_thresh)
print()
print(classification_report(y_val, y_pred_thresh,
      target_names=["No tsunami", "Tsunami"], digits=3, zero_division=0))

#### Visualize: our model vs the baselines

Bar chart comparing all three approaches. Watch how the accuracy bar is *highest* for the useless baseline, but our recall bar is much higher — proving why we don't trust accuracy alone.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

results = {
    "Always 'no tsunami'":   {"acc": accuracy_score(y_val, y_baseline_zero),
                              "prec": precision_score(y_val, y_baseline_zero, zero_division=0),
                              "rec":  recall_score(y_val, y_baseline_zero, zero_division=0),
                              "f1":   f1_score(y_val, y_baseline_zero, zero_division=0)},
    "Rule: M≥7, depth<70":   {"acc": accuracy_score(y_val, y_baseline_rule),
                              "prec": precision_score(y_val, y_baseline_rule, zero_division=0),
                              "rec":  recall_score(y_val, y_baseline_rule, zero_division=0),
                              "f1":   f1_score(y_val, y_baseline_rule, zero_division=0)},
    "Logistic Regression":   {"acc": accuracy_score(y_val, y_pred),
                              "prec": precision_score(y_val, y_pred, zero_division=0),
                              "rec":  recall_score(y_val, y_pred, zero_division=0),
                              "f1":   f1_score(y_val, y_pred, zero_division=0)},
}

metrics = ["acc", "prec", "rec", "f1"]
labels  = ["Accuracy", "Precision", "Recall", "F1"]
x = np.arange(len(metrics))
width = 0.25
colours = ["#95a5a6", "#f39c12", "#27ae60"]

for i, (name, r) in enumerate(results.items()):
    vals = [r[m] for m in metrics]
    ax.bar(x + i*width, vals, width, label=name, color=colours[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title("Our ML Model vs Baselines (Validation Set)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("→ Notice: accuracy is HIGHEST for 'always predict no tsunami'")
print("  but that model has ZERO recall — it catches no real tsunamis!")
print("→ For tsunami warnings, RECALL is the bar we care about most.")

#### Confusion matrix

A confusion matrix shows the four possible outcomes:

- **True Negatives** (top-left): correctly predicted no tsunami
- **True Positives** (bottom-right): correctly predicted tsunami
- **False Negatives** (bottom-left): missed real tsunami (the dangerous error)
- **False Positives** (top-right): false alarm

In [ ]:
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(6, 4.5))
disp = ConfusionMatrixDisplay(cm, display_labels=["No Tsunami", "Tsunami"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Logistic Regression on Validation Set")
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (correctly said no tsunami):  {tn:>4d}")
print(f"False Positives (false alarms):               {fp:>4d}")
print(f"False Negatives (MISSED tsunamis — bad):      {fn:>4d}")
print(f"True Positives  (correctly caught tsunami):   {tp:>4d}")

#### Reflection 2

> **Q4.** Comparing Logistic Regression to the "always no tsunami" baseline, which metric changed the most? Why is that improvement important for a warning system?
>
> **Q5.** From the confusion matrix: how many real tsunamis did the model miss (false negatives)? How many false alarms did it produce? For a warning system, which type of error would you most want to reduce further?
>
> **Q6.** From the interactive threshold cell above: what threshold would you actually recommend for a real tsunami warning system? Justify your choice.

**Q4:**

**Q5:**

**Q6:**

---
#### **Understanding Overfitting**

> **Learning objective:** *Understand overfitting and possible fixes.*

#### What is overfitting?

**Overfitting** happens when a model learns the training data *too well* — including the random noise and quirks. It performs beautifully on data it has seen, but poorly on new data.

Think of a student who memorizes exam answers instead of understanding the material. They ace practice tests but fail the real exam.

#### Seeing it happen with your own eyes

The easiest way to *see* overfitting is with a **decision tree** — a model that keeps splitting the data until it gets every training example right. If we let it grow too deep, it memorizes the training set perfectly but fails on new data.

We'll train decision trees at many different depths and compare training vs validation F1. The gap between them is the overfitting signal.

In [ ]:
depths = [1, 2, 3, 5, 7, 10, 15, 20, None]  # None means unlimited depth
train_f1 = []
val_f1   = []

for depth in depths:
    tree = DecisionTreeClassifier(
        max_depth=depth,
        class_weight='balanced',
        random_state=SEED,
    )
    tree.fit(X_train, y_train)
    train_f1.append(f1_score(y_train, tree.predict(X_train)))
    val_f1.append  (f1_score(y_val,   tree.predict(X_val)))

print(f"{'Depth':>7s}   {'Train F1':>9s}   {'Val F1':>7s}   {'Gap':>6s}")
print("-" * 42)
for d, tf, vf in zip(depths, train_f1, val_f1):
    d_str = "None" if d is None else str(d)
    gap = tf - vf
    marker = "  ← overfitting" if gap > 0.3 else ""
    print(f"{d_str:>7s}   {tf:9.3f}   {vf:7.3f}   {gap:6.3f}{marker}")

In [ ]:
depth_labels = [str(d) if d is not None else "None" for d in depths]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(depths)), train_f1, marker='o', ms=8,
        color='#3498db', lw=2, label='Training F1')
ax.plot(range(len(depths)), val_f1, marker='s', ms=8,
        color='#e74c3c', lw=2, label='Validation F1')

ax.fill_between(range(len(depths)), train_f1, val_f1,
                alpha=0.15, color='red', label='Overfitting gap')

best_idx = int(np.argmax(val_f1))
ax.axvline(best_idx, color='green', ls='--', alpha=0.6,
           label=f'Best depth = {depth_labels[best_idx]}')

ax.set_xticks(range(len(depths)))
ax.set_xticklabels(depth_labels)
ax.set_xlabel("Tree depth (complexity)")
ax.set_ylabel("F1 score")
ax.set_title("Overfitting: Training vs Validation Performance")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

#### What just happened?

Look at the pattern:
- **Shallow trees** (depth 1–2): both training and validation F1 are modest — the model is **underfitting** (too simple to capture the patterns)
- **Medium trees** (depth 3–5): training F1 rises, validation F1 rises too — the **sweet spot**
- **Deep trees** (depth 15+): training F1 approaches 1.0 (memorising the training set) while validation F1 stagnates or falls — **overfitting**

#### How to prevent overfitting

Some standard techniques:

1. **Limit model complexity** — set a maximum tree depth, use fewer parameters
2. **Regularisation** — mathematically penalise the model for being too complex (built into logistic regression)
3. **More training data** — makes memorisation harder
4. **Ensembling** — random forests combine many shallow trees to reduce overfitting
5. **Early stopping** — stop training when validation performance stops improving

Which technique to use depends on the model and problem.

#### Try it yourself: pick a tree depth

Now you pick a depth and see how the tree performs on both training and validation data. **Your goal:** find the depth that gives the best validation F1 without a large training-validation gap.

In [ ]:
# ── CHANGE THIS VALUE AND RE-RUN ─────────────────────────────────────────
YOUR_DEPTH = 5      # Try 1, 3, 5, 10, 20, or None (no limit)
# ────────────────────────────────────────────────────────────────────────────

your_tree = DecisionTreeClassifier(
    max_depth=YOUR_DEPTH,
    class_weight='balanced',
    random_state=SEED,
)
your_tree.fit(X_train, y_train)

train_pred = your_tree.predict(X_train)
val_pred   = your_tree.predict(X_val)

train_f1_your = f1_score(y_train, train_pred)
val_f1_your   = f1_score(y_val, val_pred)
gap = train_f1_your - val_f1_your

print(f"Decision tree with max_depth = {YOUR_DEPTH}")
print("=" * 55)
print(f"  Training F1:    {train_f1_your:.3f}")
print(f"  Validation F1:  {val_f1_your:.3f}")
print(f"  Gap:            {gap:.3f}")

if gap > 0.3:
    print("\n    Big gap → the tree is OVERFITTING")
    print("       Try a smaller depth.")
elif train_f1_your < 0.3:
    print("\n    Both F1 values are low → the tree is UNDERFITTING")
    print("       Try a larger depth.")
else:
    print("\n  The training and validation F1 are close and reasonable.")

#### Reflection 3

> **Q7.** From the plot: at what depth does the model start to overfit? (Look for where the training F1 keeps rising while validation F1 flattens or drops.)
>
> **Q8.** A colleague brings you an amazing tsunami model with 99% training accuracy. Before agreeing it's great, what questions would you ask?

**Q7:**

**Q8:**

---
#### **Challenge: Can You Improve the Model?**

> The best way to learn ML is to try to beat a baseline.

You've built a Logistic Regression model and seen how decision trees overfit. Now it's **your turn** to try improving the F1 score on the validation set. The cell below is a starting point — modify it to try new ideas.

#### Some strategies to explore

Here are a few strategies you might try, from easiest to hardest:

1. **Adjust the decision threshold** (Part 9 taught you this). Different thresholds give different precision/recall trade-offs. Which one maximises F1?

2. **Try a Random Forest.** A random forest is an ensemble of many decision trees — it tends to be more accurate and less prone to overfitting than a single tree. Try `RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)`.

3. **Change model hyperparameters.** For Random Forest, try `max_depth`, `min_samples_leaf`, or `n_estimators`. For Logistic Regression, try `C` (the regularisation strength — smaller values mean more regularisation).

4. **Add or remove features.** Would `mag * shallow` (an interaction term) help? What if you dropped features that seem noisy like `dmin` or `rms`?

5. **Try a different rebalancing approach.** Instead of `class_weight='balanced'`, try oversampling the minority class with `imbalanced-learn` (which you'd need to install).

Track your best score on the validation set. Only once you're happy with your model should you test it on the test set — and only once!

In [ ]:
# ── YOUR CHALLENGE — try to beat this baseline ───────────────────────────
# Current baseline to beat (Logistic Regression, default threshold):
BASELINE_F1 = f1_score(y_val, y_pred)
print(f"Baseline F1 (Logistic Regression at threshold 0.5): {BASELINE_F1:.3f}")
print(f"Your goal: beat this on the validation set.\n")

# ─── STARTING POINT: A Random Forest with default settings ──────────────────
# Try tweaking the parameters below, or replace this with your own model!

my_model = RandomForestClassifier(
    n_estimators=100,       # try: 50, 200, 500
    max_depth=None,         # try: 5, 10, 20, None
    min_samples_leaf=1,     # try: 1, 5, 10, 20
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
)
my_model.fit(X_train, y_train)

my_pred  = my_model.predict(X_val)
my_proba = my_model.predict_proba(X_val)[:, 1]

my_f1 = f1_score(y_val, my_pred)
my_auc = roc_auc_score(y_val, my_proba)

print(f"Your model's F1:  {my_f1:.3f}    (baseline: {BASELINE_F1:.3f})")
print(f"Your model's AUC: {my_auc:.3f}\n")

if my_f1 > BASELINE_F1:
    print(f"Great! You beat the baseline by {(my_f1 - BASELINE_F1):.3f}!")
else:
    print("Not quite there yet. Try adjusting parameters or a different strategy.")

print("\nDetails:")
print_metrics("Your model", y_val, my_pred)


#### **Final Evaluation on Test Set**

We used the validation set to compare models and tune hyperparameters. Now — and only now — we look at the test set. This is our unbiased estimate of how the model would perform on truly new data.

**Rule:** we do not go back and change anything based on test performance. If we did, we'd effectively be training on the test set too.

In [ ]:
# ── Train the best tree using validation results to choose depth ────────────
BEST_DEPTH = depths[int(np.argmax(val_f1))]
final_tree = DecisionTreeClassifier(
    max_depth=BEST_DEPTH,
    class_weight='balanced',
    random_state=SEED,
)
final_tree.fit(X_train, y_train)

# ── Evaluate all our models on the TEST set ─────────────────────────────────
lr_test_pred     = model.predict(X_test_scaled)
tree_test_pred   = final_tree.predict(X_test)
mymodel_test_pred = my_model.predict(X_test)   # your challenge model
rule_test        = ((X_test['mag'] >= 7.0) & (X_test['shallow'] == 1)).astype(int).values

print("FINAL TEST SET PERFORMANCE")
print("=" * 95)
print_metrics("Rule baseline",                            y_test, rule_test)
print_metrics("Logistic Regression",                      y_test, lr_test_pred)
print_metrics(f"Decision Tree (best depth={BEST_DEPTH})", y_test, tree_test_pred)
print_metrics("Your challenge model",                     y_test, mymodel_test_pred)


#### **Interpret and Discuss Limitations**

#### What did the model actually learn?

For logistic regression, we can look directly at the model coefficients to see which features drive its predictions.

In [ ]:
coefs = pd.Series(model.coef_[0], index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colours_c = ["#e74c3c" if v > 0 else "#3498db" for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colours_c, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title("What the Logistic Regression Learned\n(red = raises tsunami probability, blue = lowers it)")
ax.set_xlabel("Standardized coefficient")
plt.tight_layout(); plt.show()

print("Top predictors of tsunami:")
for feat, val in coefs.sort_values(ascending=False).head(3).items():
    print(f"  ↑ {feat:20s} coefficient = {val:+.3f}")

**Sanity check:** the top predictors should be `mag`, `shallow`, and/or location features. This matches what seismologists know — big shallow offshore earthquakes generate tsunamis. If the model relied on features that had nothing to do with the physics (like number of stations), that would be a red flag.

#### Limitations of our model

Understanding what the model *can't* do is as important as understanding what it can.

**Data limitations:**
- We used only the standard catalog. We do not have **bathymetry** (ocean depth at the epicentre) — a huge factor in tsunami generation.
- We do not have **focal mechanism** (whether fault motion was vertical or horizontal). Vertical motion generates tsunamis; horizontal typically does not.
- The `tsunami` label was assigned **retrospectively** by USGS analysts. A real warning system does not have this luxury.
- Small earthquakes in remote regions may be **incomplete** in the catalog.

**Modelling limitations:**
- Our model is trained on the entire globe. It might work poorly for specific regions (e.g. unusual tectonic settings).
- Tsunamis can also be caused by **submarine landslides, volcanic eruptions, and meteor impacts** — none of which are in our data.

#### Responsible use

Before *any* real-world use, a system like this needs:

- **Human expert review** — a qualified seismologist must validate alerts before public release
- **Real-time sensor integration** — DART buoys, tide gauges, and hydrodynamic models
- **Community engagement** — affected coastal populations should have input on alert thresholds
- **Equity analysis** — the model's errors don't affect everyone equally. Missed warnings hurt those with least evacuation capacity most.
- **Multi-source verification** — never trust a single model
- **Alert fatigue mitigation** — repeated false alarms train people to ignore warnings, a real and documented problem

#### Reflection 4

> **Q9.** Suppose a coastal government adopts a model with 90% recall and 30% precision. Over a year with 100 real tsunamis and 3,000 non-tsunami earthquakes above the M4.5 threshold, roughly how many false alarms will it issue? How many real tsunamis will it miss?
>
> **Q10.** A colleague proposes adding bathymetry (ocean depth) and focal mechanism as features. Where would you get this data? What new risks might they introduce?
>
> **Q11.** The model was trained on 2015-2024 earthquakes. What could go wrong if a completely novel type of event occurs — for example, an M9+ earthquake in a region with no M7+ events in the training set?

**Q9:**

**Q10:**

**Q11:**

---
#### **Extra Challenges (Optional)**

If you finished early or want to go deeper, try these.

In [ ]:
# ── Challenge A: Filter to a region relevant to you ─────────────────────────
# Refetch data for just a region you care about:
#   Japan:      lat 25-45,  lon 125-150
#   Chile:      lat -55--18, lon -76--66
#   Indonesia:  lat -11-6,  lon 95-141
#   Turkey:     lat 36-42,  lon 26-45
# Rebuild the model with just that region's data.
# How does model performance change? Why might it differ?

# Your code here:

In [ ]:
# ── Challenge B: Predict magnitude instead ───────────────────────────────────
# Instead of classifying tsunami/no-tsunami, try REGRESSION: predict earthquake
# magnitude from the other features.
#
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, r2_score
#
# X_reg = df_clean[['depth','shallow','nst','gap','horizontalError','depthError']]
# y_reg = df_clean['mag']

# Your code here:

------

<h2 id="summary">Summary</h2>

You've completed the full machine learning workflow using real earthquake data:

| Step | What we did |
|---|---|
| **1. Look at the data** | Fetched from USGS API; visualised global patterns with an interactive map |
| **2. Define features & target** | Chose magnitude, depth, location (cyclic-encoded), and quality features |
| **3. Train/val/test split** | Stratified 60/20/20 to preserve rare-class rate |
| **4. Baseline** | Compared to "always no" and a simple magnitude+depth rule |
| **5. Train a model** | Fit logistic regression with balanced class weights |

---


<h2 id="references">References</h2>

- USGS Earthquake Hazards Program: <https://earthquake.usgs.gov/>
- Pacific Tsunami Warning Center: <https://www.tsunami.gov/>
- Folium documentation for interactive maps: <https://python-visualization.github.io/folium/>
- STEAD dataset (ML on raw seismic waveforms): <https://github.com/smousavi05/STEAD>
- SeisBench Python toolbox: <https://github.com/seisbench/seisbench>